## Plotly

In [258]:
import pandas as pd
import sqlite3
import plotly.graph_objects as go
import numpy as np

conn = sqlite3.connect('../data/checking-logs.sqlite')
all_users = pd.read_sql("SELECT DISTINCT uid FROM checker WHERE uid LIKE 'user_%'", conn)['uid'].tolist()

query = """
    SELECT uid, timestamp
    FROM checker
    WHERE uid LIKE 'user_%' AND labname = 'project1' AND status = 'ready'
"""
df = pd.read_sql(query, conn, parse_dates=['timestamp'])
conn.close()

## data processing

In [259]:
df.sort_values('timestamp', inplace=True)
df['commit_count'] = 1
df['num_commits'] = df.groupby('uid')['commit_count'].cumsum()
df.head()

,uid,timestamp,commit_count,num_commits
0,user_4,2020-04-17 05:19:02.744528,1,1
1,user_4,2020-04-17 05:22:45.549397,1,2
2,user_4,2020-04-17 05:34:24.422370,1,3
3,user_4,2020-04-17 05:43:27.773992,1,4
4,user_4,2020-04-17 05:46:32.275104,1,5


In [260]:
df_wide = df.pivot_table(index='timestamp', columns='uid', values='num_commits')
df_wide = df_wide.resample('D').max().ffill().fillna(0)
df_wide = df_wide.reindex(columns=all_users, fill_value=0)

# transponation
df_wide = df_wide.T 

# parse dates to package like steps of animations
dates = df_wide.columns
users = df_wide.index
n_frames = len(dates)
x_numeric = np.arange(n_frames)

## make graph like hindu in git hub

In [261]:
# first day
initial_data = []
for user in users:
    initial_data.append(go.Scatter(
        x=[0], 
        y=[df_wide.iloc[df_wide.index.get_loc(user), 0]], 
        mode="lines+markers", 
        marker=dict(size=8),
        name=user,
        line=dict(width=3)
    ))
# more frames
frames = []
for f in range(1, n_frames):
    curr_data = []
    for user in users:
        # from 0 to current 
        y_axis = df_wide.loc[user].iloc[:f+1]
        x_axis = x_numeric[:f+1]
        
        curr_data.append(go.Scatter(
            x=x_axis, 
            y=y_axis, 
            mode="lines+markers", 
            name=user,
            line=dict(width=3),
            marker=dict(size=8)
        ))
    # frame add to list
    frames.append(go.Frame(data=curr_data, name=str(f)))

## final figure

In [262]:
fig = go.Figure(
    data=initial_data,
    layout={
        "title": {"text": "Dynamic of commits per user in project1", 
            "font": {"size": 17},
            "x": 0.04,
            "xanchor": "left",
            "yanchor": "top"
        },
        "template":"ggplot2", 
        "xaxis": {
            "title": None,
            "range": [0, n_frames], 
            "dtick": 2,
            "visible": True, 
            "showgrid": True,
        },
        "yaxis": {
            "range": [0, 165],
            "dtick": 20,
            "visible": True, 
            "showgrid": True,
        },
        "updatemenus": [{
            "type": "buttons",
            "buttons": [{
                "label": "play",
                "method": "animate",
                "args": [None, {"frame": {"duration": 100, "redraw": True}, "fromcurrent": True}]
            }]
        }]
    },
    frames=frames
)

fig.show()